# Chat Mode — Integration Test

Tests the full chat pipeline across multiple questions:
- **Q1:** "How frequent were positive events (e.g. limit increases) in the last 18 months?"
- **Follow-up:** "Are the positive events consistent with the bureau scores?" (brings in bureau)
- **Q2 (independent):** "What does the recent 3 months payment pattern look like?"

In [ ]:
%load_ext dotenv
%dotenv

%load_ext autoreload
%autoreload 2

import os, sys, json
import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')
PILLAR_DIR = os.path.join(PROJECT_ROOT, 'config', 'pillars')

# ═══════════════ CONFIG ═══════════════
SAFECHAIN_MODEL = "gpt-4.1"
CASE_ID = "CASE-00001"
PILLAR = "credit_risk"
MODE = "chat"
QUESTION = "How frequent were positive events (e.g. limit increases) in the last 18 months?"
# ═════════════════════════════════════

print(f'Case: {CASE_ID} | Pillar: {PILLAR} | Mode: {MODE}')
print(f'Question: {QUESTION}')

## 1. Load Data

In [ ]:
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools

gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)
gw.set_case(CASE_ID)

print(f'Tables: {gw.list_tables()}')

## 2. Preview Relevant Data

In [ ]:
print('All model_scores rows (sorted by trans_month, descending):')
rows = gw.query('model_scores')
if rows:
    sorted_rows = sorted(rows, key=lambda r: r.get('trans_month', ''), reverse=True)
    for row in sorted_rows:
        pe = row.get('positive_events', 'N/A')
        marker = '  <-- POSITIVE EVENT' if str(pe) not in ('0', 'N/A') else ''
        print(f'  {row.get("trans_month")}: positive_events={pe}{marker}')

print()
tenure = gw.query('cust_tenure')
if tenure:
    print(f'Tenure: {tenure[0]}')

## 3. Connect LLM

In [ ]:
from gateway.safechain_adapter import SafeChainAdapter

try:
    from safechain.lcel import model as safechain_model
    llm = safechain_model(SAFECHAIN_MODEL)
    adapter = SafeChainAdapter(llm=llm, model_name=SAFECHAIN_MODEL)
    print(f'SafeChain adapter: model={SAFECHAIN_MODEL}')
except ImportError:
    from gateway.openai_adapter import OpenAIAdapter
    adapter = OpenAIAdapter(model='gpt-4.1')
    print(f'OpenAI adapter (SafeChain not available)')

## 4. Pipeline Setup

In [ ]:
from gateway.firewall_stack import FirewallStack
from logger.event_logger import EventLogger
from agents.session_registry import SessionRegistry
from agents.general_specialist import GeneralSpecialist
from orchestrator.orchestrator import Orchestrator
from orchestrator.chat_agent import ChatAgent
from config.pillar_loader import PillarLoader
from skills.domain.loader import load_domain_skill, list_domain_skills

logger = EventLogger(session_id=f'chat-{CASE_ID}', log_dir=os.path.join(PROJECT_ROOT, 'logs'))
pillar_loader = PillarLoader(pillar_dir=PILLAR_DIR)
pillar_config = pillar_loader.load(PILLAR) or {}
registry = SessionRegistry()

# Orchestrator owns team planning (specialist selection + sub-question split)
# and synthesis. Pass catalog so it can describe per-specialist data coverage.
general = GeneralSpecialist(firewall=FirewallStack(adapter=adapter, logger=logger), logger=logger)
orchestrator = Orchestrator(
    FirewallStack(adapter=adapter, logger=logger), logger, registry, PILLAR,
    pillar_config=pillar_config, catalog=catalog,
)
chat_agent = ChatAgent(firewall=FirewallStack(adapter=adapter, logger=logger), logger=logger)

logger.log('session_start', {'case_id': CASE_ID, 'pillar': PILLAR, 'mode': MODE})
print('Pipeline initialized')

## 5. Helper Function

In [ ]:
from IPython.display import Markdown, display


def run_question(question: str, trace_id: str):
    """Run one question through: plan team (select + split) -> specialists -> compare -> synthesize."""
    logger.set_trace(trace_id)

    # Team planning — one LLM call returns specialist selection + per-specialist sub-questions.
    plan = orchestrator.plan_team(
        question=question,
        available_specialists=list_domain_skills(),
        active_specialists=registry.list_active(),
        mode=MODE,
    )
    print(f'Plan ({len(plan)} specialist(s)):')
    for a in plan:
        print(f'  - {a.specialist}: {a.sub_question}')
    print(f'(Warm: {[a["domain"] for a in registry.list_active()]})')
    print()

    # Run specialists (get_or_create reuses warm instances) — each gets its sub-question + root.
    outputs = {}
    for assignment in plan:
        skill = load_domain_skill(assignment.specialist)
        if skill is None:
            continue
        spec_config = pillar_loader.get_specialist_config(PILLAR, assignment.specialist) or {}
        fw = FirewallStack(adapter=adapter, logger=logger)
        agent = registry.get_or_create(assignment.specialist, PILLAR, skill, spec_config, fw, logger)
        output = agent.run(assignment.sub_question, mode=MODE, root_question=question)
        outputs[assignment.specialist] = output
        print(f'  {assignment.specialist}: {output.findings[:200]}...')
    print()

    # Compare + synthesize
    review = general.compare(outputs, question)
    final = orchestrator.synthesize(outputs, review, question, MODE, team_plan=plan)

    logger.log('final_output', {'question': question, 'specialists': final.specialists_consulted})
    logger.clear_trace()
    return plan, outputs, review, final


def display_answer(question: str, plan, outputs, final):
    """Render the final output as Markdown with all pipeline details surfaced.

    Shows: final answer (bullet-structured), specialists consulted (selected + warm),
    sub-questions per specialist, per-specialist findings, cross-domain insights,
    resolved contradictions, data requests made during review, open conflicts,
    data-gap summary + signal gaps, and any incomplete analyses.
    """
    selected = [a.specialist for a in plan]
    formatted = chat_agent.format_for_reviewer(
        final,
        specialist_outputs=outputs,
        selected=selected,
        warm=registry.list_active(),
    )
    md = f"### Question\n\n{question}\n\n---\n\n{formatted}"
    display(Markdown(md))

## 6. Q1 — Positive Events Frequency

In [ ]:
plan_q1, outputs_q1, review_q1, final_q1 = run_question(QUESTION, 'q-001')
display_answer(QUESTION, plan_q1, outputs_q1, final_q1)

## 7. Follow-Up — Brings in Bureau

"Are the positive events consistent with the bureau scores?" — this question needs BOTH modeling (positive_events) AND bureau (scores). The team constructor should pull in bureau on top of the warm modeling specialist from Q1.

In [ ]:
print('=== Data for follow-up validation ===')
print()

print('Bureau scores (from bureau table):')
bureau_rows = gw.query('bureau')
if bureau_rows:
    for row in bureau_rows[:2]:
        print(f'  fico={row.get("fico_score")}, sbfe={row.get("sbfe_score")}, ln_credit={row.get("ln_credit_score")}, paydex={row.get("paydex_score")}')

print()
print('Positive events timeline (model_scores, sorted by trans_month):')
model_rows = gw.query('model_scores')
if model_rows:
    sorted_rows = sorted(model_rows, key=lambda r: r.get('trans_month', ''))
    for row in sorted_rows:
        pe = row.get('positive_events', 'N/A')
        marker = '  <-- POSITIVE EVENT' if str(pe) not in ('0', 'N/A') else ''
        print(f'  {row.get("trans_month")}: positive_events={pe}{marker}')

In [ ]:
FOLLOW_UP = "Are the positive events consistent with the bureau scores?"
plan_fu, outputs_fu, review_fu, final_fu = run_question(FOLLOW_UP, 'q-002')
display_answer(FOLLOW_UP, plan_fu, outputs_fu, final_fu)

## 8. Q2 — Independent Topic (Recent Payment Pattern)

A completely different question — not related to positive events. The team constructor should select spend_payments, possibly bureau or crossbu.

In [ ]:
CUT_OFF = pillar_config.get('cut_off_date', '2025-12-01')
print(f'=== Data for Q2 validation (cut-off: {CUT_OFF}) ===')
print()

print('Monthly transactions (txn_monthly, sorted recent first):')
txn = gw.query('txn_monthly')
if txn:
    sorted_txn = sorted(txn, key=lambda r: r.get('month', ''), reverse=True)
    for row in sorted_txn[:5]:
        print(f'  {row.get("month")}: spend_total=${row.get("spend_total")}, txn_count={row.get("txn_count")}, category={row.get("category")}')

print()
print('Recent spend transactions (spends, last 5 by date):')
spends = gw.query('spends')
if spends:
    sorted_spends = sorted(spends, key=lambda r: r.get('spend_date', ''), reverse=True)
    for row in sorted_spends[:5]:
        print(f'  {row.get("spend_date")}: ${row.get("amount")} @ {row.get("merchant_name")}, risk={row.get("merchant_risk_score")}')

print()
print('Recent payments (payments, last 10 by date):')
pmts = gw.query('payments')
if pmts:
    sorted_pmts = sorted(pmts, key=lambda r: r.get('payment_date', ''), reverse=True)
    for row in sorted_pmts[:10]:
        status = row.get('return_flag')
        reason = f' -- {row.get("return_reason")}' if status == 'returned' else ''
        print(f'  {row.get("payment_date")}: ${row.get("payment_amount")} card={row.get("card_number")} [{status}]{reason}')

In [ ]:
QUESTION_2 = "What does the recent 3 months payment pattern look like? Any signs of deterioration?"
plan_q2, outputs_q2, review_q2, final_q2 = run_question(QUESTION_2, 'q-003')
display_answer(QUESTION_2, plan_q2, outputs_q2, final_q2)

## 9. Session Info

In [ ]:
print('Active specialists across the session:')
for a in registry.list_active():
    print(f'  {a["domain"]}: {a["questions_answered"]} questions answered')

print()
print(f'Q1 plan:        {[a.specialist for a in plan_q1]}')
print(f'Follow-up plan: {[a.specialist for a in plan_fu]}')
print(f'Q2 plan:        {[a.specialist for a in plan_q2]}')

logger.log('session_end', {})
print('\nSession complete.')